In [1]:
import pandas as pd

train_df = pd.read_csv('../dataframes/two_input_train_df_cleaned.csv')
test_df = pd.read_csv('../dataframes/two_input_test_df_cleaned.csv')
val_df = pd.read_csv('../dataframes/two_input_val_df_cleaned.csv')

In [2]:
from pathlib import Path

ROOT_DIR = Path.cwd().parent / 'cbis_ddsm_png'
CACHE_DIR = Path.cwd().parent / 'cbis-ddsm-cache-optimized'

In [5]:
import hashlib
import os
from pathlib import Path

from PIL import Image

# ============================================================
# Windows Long Path Helper
# ============================================================

def windows_long_path(path):
    """
    Adds the Windows long-path prefix if needed.
    """
    path = str(Path(path).resolve())

    if os.name == "nt" and not path.startswith("\\\\?\\"):
        path = "\\\\?\\" + path

    return path


# ============================================================
# Preprocess: aspect-ratio-preserving resize
# ============================================================

def preprocess_image(image_path, size=(224, 224)):
    """
    Opens an image, converts it to RGB, and resizes it to `size`
    WITHOUT distorting aspect ratio: the image is scaled down to fit
    inside `size` (LANCZOS) and then centered on a black canvas of
    exactly `size`.

    This matters for mammography specifically: mass shape and margin
    are diagnostic features, and a naive stretch-to-square resize can
    distort exactly the signal the model needs to learn from, especially
    for the cropped-lesion branch.
    """
    img = Image.open(windows_long_path(image_path)).convert("RGB")

    img = img.copy()
    img.thumbnail(size, Image.Resampling.LANCZOS)

    canvas = Image.new("RGB", size, (0, 0, 0))
    offset = ((size[0] - img.width) // 2, (size[1] - img.height) // 2)
    canvas.paste(img, offset)

    return canvas


# ============================================================
# Stable cache key
# ============================================================

def _cache_key(source_path):
    """
    Derives a filename-safe key from the SOURCE path rather than the row's
    positional index. This makes caching:
      - stable across re-runs / row reordering (same source -> same file)
      - safe across splits, since the key depends only on the source path,
        never on which split or index the row happened to have
    """
    return hashlib.md5(str(source_path).encode("utf-8")).hexdigest()[:16]


# ============================================================
# Cache Dataset
# ============================================================

def cache_dataset(df, root_dir, cache_dir, size=(224, 224), split_name=None):
    """
    Saves resized PNGs to:

    cache_dir/
        full/
            <hash>.png
            ...
        crop/
            <hash>.png
            ...

    IMPORTANT: pass a SEPARATE `cache_dir` per split (e.g. CACHE_DIR /
    "train", CACHE_DIR / "val", CACHE_DIR / "test") so that caching
    multiple splits can never collide with each other -- the original
    version keyed files by positional row index and reset that index
    per split, so val/test rows silently reused train's cached images
    whenever indices matched. Using content-hashed filenames (see
    _cache_key) plus a distinct cache_dir per split closes that gap
    even if two splits happen to be cached into sibling folders.

    `split_name` is optional and used only for log messages -- if not
    given, it defaults to `cache_dir`'s own folder name (e.g. passing
    CACHE_DIR / "val" as cache_dir will log as "val" automatically).

    Rows that fail to process are DROPPED from the returned dataframe
    (previously they were kept with an empty-string cached path, which
    crashed training later with FileNotFoundError instead of failing
    fast here, where it's obvious what went wrong).

    Returns (enriched_df, failed_row_info) where enriched_df contains
    only successfully cached rows, and failed_row_info is a list of
    (original_index, error_message) for anything that failed.
    """
    cache_dir = Path(cache_dir)
    split_name = split_name or cache_dir.name

    full_cache = cache_dir / "full"
    crop_cache = cache_dir / "crop"

    full_cache.mkdir(parents=True, exist_ok=True)
    crop_cache.mkdir(parents=True, exist_ok=True)

    enriched_df = df.copy().reset_index(drop=True)
    enriched_df["full image cached path"] = ""
    enriched_df["cropped image cached path"] = ""

    failed = []
    total = len(enriched_df)

    for idx, row in enriched_df.iterrows():
        try:
            full_path = Path(root_dir) / row["image file path"]
            crop_path = Path(root_dir) / row["cropped image file path"]

            full_key = _cache_key(full_path)
            crop_key = _cache_key(crop_path)

            full_output = full_cache / f"{full_key}.png"
            crop_output = crop_cache / f"{crop_key}.png"

            # -------------------------
            # Full Mammogram
            # -------------------------
            if not full_output.exists():
                img = preprocess_image(full_path, size)
                img.save(windows_long_path(full_output), format="PNG", optimize=True)

            # -------------------------
            # Cropped Mammogram
            # -------------------------
            if not crop_output.exists():
                img = preprocess_image(crop_path, size)
                img.save(windows_long_path(crop_output), format="PNG", optimize=True)

            enriched_df.at[idx, "full image cached path"] = str(full_output)
            enriched_df.at[idx, "cropped image cached path"] = str(crop_output)

            if (idx + 1) % 100 == 0:
                print(f"[{split_name}] {idx + 1}/{total} cached")

        except Exception as e:
            print(f"\n[{split_name}] Failed on row {idx}")
            print(e)
            failed.append((idx, str(e)))

    # Drop failed rows so downstream training code never sees an empty
    # cached-path string -- previously these rows were kept and would
    # raise FileNotFoundError deep inside a training run instead of here.
    if failed:
        failed_indices = [i for i, _ in failed]
        enriched_df = enriched_df.drop(index=failed_indices).reset_index(drop=True)

    print(f"\n{'=' * 30}")
    print(f"Caching Finished: {split_name}")
    print(f"{'=' * 30}")
    print(f"Total Samples : {total}")
    print(f"Cached        : {total - len(failed)}")
    print(f"Failed        : {len(failed)}")

    return enriched_df, failed


# ============================================================
# Convenience wrapper for the standard train/val/test split
# ============================================================

def cache_all_splits(train_df, val_df, test_df, root_dir, cache_dir, size=(224, 224)):
    """
    Caches all three splits into namespaced subfolders of `cache_dir` and
    returns (train_df, val_df, test_df, failed_report) with the enriched,
    failure-filtered dataframes ready to hand to DualImageDataset.
    """
    cache_dir = Path(cache_dir)
    train_cached, train_failed = cache_dataset(train_df, root_dir, cache_dir / "train", size)
    val_cached, val_failed = cache_dataset(val_df, root_dir, cache_dir / "val", size)
    test_cached, test_failed = cache_dataset(test_df, root_dir, cache_dir / "test", size)

    failed_report = {
        "train": train_failed,
        "val": val_failed,
        "test": test_failed,
    }
    return train_cached, val_cached, test_cached, failed_report

In [2]:
import pandas as pd

def check_leakage(train_df, val_df, test_df, id_col="patient_id"):
    t, v, te = set(train_df[id_col]), set(val_df[id_col]), set(test_df[id_col])
    print("train∩val :", len(t & v))
    print("train∩test:", len(t & te))
    print("val∩test  :", len(v & te))

check_leakage(train_df, val_df, test_df)  # also try id_col="image file path"

train∩val : 0
train∩test: 0
val∩test  : 0


In [ ]:
# train_df, train_failed = cache_dataset(train_df, ROOT_DIR, CACHE_DIR / "train")
# val_df, val_failed = cache_dataset(val_df, ROOT_DIR, CACHE_DIR / "val")
# test_df, test_failed = cache_dataset(test_df, ROOT_DIR, CACHE_DIR / "test")

# print("\nSummary")
# print(f"Train failed: {len(train_failed)}")
# print(f"Val failed  : {len(val_failed)}")
# print(f"Test failed : {len(test_failed)}")

[train] 100/2599 cached
[train] 200/2599 cached
[train] 300/2599 cached
[train] 400/2599 cached
[train] 500/2599 cached
[train] 600/2599 cached
[train] 700/2599 cached
[train] 800/2599 cached
[train] 900/2599 cached
[train] 1000/2599 cached
[train] 1100/2599 cached
[train] 1200/2599 cached
[train] 1300/2599 cached
[train] 1400/2599 cached
[train] 1500/2599 cached
[train] 1600/2599 cached
[train] 1700/2599 cached
[train] 1800/2599 cached
[train] 1900/2599 cached
[train] 2000/2599 cached
[train] 2100/2599 cached
[train] 2200/2599 cached
[train] 2300/2599 cached
[train] 2400/2599 cached
[train] 2500/2599 cached

Caching Finished: train
Total Samples : 2599
Cached        : 2599
Failed        : 0
[val] 100/328 cached
[val] 200/328 cached
[val] 300/328 cached

Caching Finished: val
Total Samples : 328
Cached        : 328
Failed        : 0
[test] 100/326 cached
[test] 200/326 cached
[test] 300/326 cached

Caching Finished: test
Total Samples : 326
Cached        : 326
Failed        : 0

Summar

In [ ]:
train_df.columns.tolist()

['patient_id',
 'pathology',
 'image file path',
 'cropped image file path',
 'full image cached path',
 'cropped image cached path']

In [ ]:
test_df.columns.tolist()

['patient_id',
 'pathology',
 'image file path',
 'cropped image file path',
 'full image cached path',
 'cropped image cached path']

In [ ]:
val_df.columns.tolist()

['patient_id',
 'pathology',
 'image file path',
 'cropped image file path',
 'full image cached path',
 'cropped image cached path']

In [ ]:
pd.set_option("display.max_colwidth", None)


train_df.head()


,patient_id,pathology,image file path,cropped image file path,full image cached path,cropped image cached path
0,P_00007,0,Calc-Training_P_00007_LEFT_CC/1.3.6.1.4.1.9590.100.1.2.201322325113694962619881476352450072222/1.3.6.1.4.1.9590.100.1.2.228699627313487111012474405462022068297/d6536189-e9ad-49c7-8195-f4159f22ca3c.png,Calc-Training_P_00007_LEFT_CC_1/1.3.6.1.4.1.9590.100.1.2.241202057913673145232234613012384759880/1.3.6.1.4.1.9590.100.1.2.314135871111943890422150247820137952041/6c7ea586-d2f5-4cc4-8f85-778279bf0a72.png,c:\Users\Drew\Documents\University\2025-2026\2nd Semester\Thesis\Experiment\cbis-ddsm-cache-indexed\train\full\full_0.png,c:\Users\Drew\Documents\University\2025-2026\2nd Semester\Thesis\Experiment\cbis-ddsm-cache-indexed\train\crop\crop_0.png
1,P_00007,0,Calc-Training_P_00007_LEFT_MLO/1.3.6.1.4.1.9590.100.1.2.370479499712916693322010643793108454887/1.3.6.1.4.1.9590.100.1.2.104743410411133110629448544090466900446/eb181cf5-3703-4280-b9e0-f38557614848.png,Calc-Training_P_00007_LEFT_MLO_1/1.3.6.1.4.1.9590.100.1.2.314250272911170289203882349024229868823/1.3.6.1.4.1.9590.100.1.2.91458279612485515203413781822560852485/82ffd830-9912-48c7-a225-8dd460aa0d34.png,c:\Users\Drew\Documents\University\2025-2026\2nd Semester\Thesis\Experiment\cbis-ddsm-cache-indexed\train\full\full_1.png,c:\Users\Drew\Documents\University\2025-2026\2nd Semester\Thesis\Experiment\cbis-ddsm-cache-indexed\train\crop\crop_1.png
2,P_00008,0,Calc-Training_P_00008_LEFT_CC/1.3.6.1.4.1.9590.100.1.2.162256682111885666305889708873412464189/1.3.6.1.4.1.9590.100.1.2.406725628213826290127343763811145520834/beb4a6cf-b56b-4ce8-a0eb-6cc093b34d4e.png,Calc-Training_P_00008_LEFT_CC_1/1.3.6.1.4.1.9590.100.1.2.336811694512764490002272925921108351157/1.3.6.1.4.1.9590.100.1.2.281397494612871934937455783843630775495/cc2ac0cd-e861-4a04-96b9-3c4ff5ce9a04.png,c:\Users\Drew\Documents\University\2025-2026\2nd Semester\Thesis\Experiment\cbis-ddsm-cache-indexed\train\full\full_2.png,c:\Users\Drew\Documents\University\2025-2026\2nd Semester\Thesis\Experiment\cbis-ddsm-cache-indexed\train\crop\crop_2.png
3,P_00008,0,Calc-Training_P_00008_LEFT_CC/1.3.6.1.4.1.9590.100.1.2.162256682111885666305889708873412464189/1.3.6.1.4.1.9590.100.1.2.406725628213826290127343763811145520834/beb4a6cf-b56b-4ce8-a0eb-6cc093b34d4e.png,Calc-Training_P_00008_LEFT_CC_2/1.3.6.1.4.1.9590.100.1.2.207558519313356140106496367610041842177/1.3.6.1.4.1.9590.100.1.2.2024181113538627406890441552979568519/1abe36a0-00b6-4466-af67-124a863b3cfa.png,c:\Users\Drew\Documents\University\2025-2026\2nd Semester\Thesis\Experiment\cbis-ddsm-cache-indexed\train\full\full_3.png,c:\Users\Drew\Documents\University\2025-2026\2nd Semester\Thesis\Experiment\cbis-ddsm-cache-indexed\train\crop\crop_3.png
4,P_00008,0,Calc-Training_P_00008_LEFT_CC/1.3.6.1.4.1.9590.100.1.2.162256682111885666305889708873412464189/1.3.6.1.4.1.9590.100.1.2.406725628213826290127343763811145520834/beb4a6cf-b56b-4ce8-a0eb-6cc093b34d4e.png,Calc-Training_P_00008_LEFT_CC_3/1.3.6.1.4.1.9590.100.1.2.74480335312443720440103727953726257477/1.3.6.1.4.1.9590.100.1.2.358964325411863896506852883280957478483/24d0b225-f083-4964-8498-11d7236790c5.png,c:\Users\Drew\Documents\University\2025-2026\2nd Semester\Thesis\Experiment\cbis-ddsm-cache-indexed\train\full\full_4.png,c:\Users\Drew\Documents\University\2025-2026\2nd Semester\Thesis\Experiment\cbis-ddsm-cache-indexed\train\crop\crop_4.png


In [7]:
train_df.to_csv("../dataframes/train_df_final_optimized.csv", index=False)
test_df.to_csv("../dataframes/test_df_final_optimized.csv", index=False)
val_df.to_csv("../dataframes/val_df_final_optimized.csv", index=False)